# 🚚 FedEx Logistics Performance Analysis

**Project Type:** Exploratory Data Analysis (EDA) | **Contribution:** Individual

**GitHub:** https://github.com/Kedarlimbalkar/FedEx-Logistics-Performance-Analysis

---

## Problem Statement

FedEx Logistics operates a vast global supply chain where inefficiencies cause costly delays. This project analyzes **10,324 shipment records across 43 countries** to identify bottlenecks, evaluate cost-effectiveness of shipment methods, and improve delivery timelines.

**Business Objective:** Optimize logistics to reduce costs, improve delivery times, and enhance customer satisfaction.

## 1. Know Your Data

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('Libraries imported successfully!')

### Dataset Loading

In [ ]:
# Load Dataset - update path as needed
FILE_PATH = 'data/raw/SCMS_Delivery_History_Dataset.csv'

dataset = pd.read_csv(FILE_PATH)
df = dataset.copy()
print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

### Dataset First View

In [ ]:
df.head()

In [ ]:
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.info()

In [ ]:
df.describe()

In [ ]:
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print(f'Total missing: {df.isnull().sum().sum():,}')

plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/missing_values_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Understanding Your Variables

In [ ]:
for i, col in enumerate(df.columns, 1):
    print(f'{i:2}. {col}')

In [ ]:
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique values')

## 3. Data Wrangling

In [ ]:
# Convert date columns
date_cols = ['PQ First Sent to Client Date','PO Sent to Vendor Date',
             'Scheduled Delivery Date','Delivered to Client Date','Delivery Recorded Date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
print('Date columns converted')

# Clean numeric columns stored as text
df['Freight Cost (USD)'] = pd.to_numeric(df['Freight Cost (USD)'], errors='coerce')
df['Weight (Kilograms)'] = pd.to_numeric(df['Weight (Kilograms)'], errors='coerce')
print('Numeric columns cleaned')

# Fill missing Shipment Mode
df['Shipment Mode'].fillna(df['Shipment Mode'].mode()[0], inplace=True)

# Feature Engineering
df['Delivery Delay (Days)'] = (df['Delivered to Client Date'] - df['Scheduled Delivery Date']).dt.days
df['Cost per Unit'] = df['Freight Cost (USD)'] / df['Line Item Quantity']
print('Features created: Delivery Delay, Cost per Unit')
print(f'Final shape: {df.shape}')

## 4. Data Visualization

### Univariate Analysis

In [ ]:
# Chart 1: Shipment Mode Distribution
# Why: Understand which transport methods dominate operations
fig, ax = plt.subplots(figsize=(8, 5))
mode_counts = df['Shipment Mode'].value_counts()
bars = ax.bar(mode_counts.index, mode_counts.values, color=sns.color_palette('husl', len(mode_counts)))
for bar, val in zip(bars, mode_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30, f'{val:,}', ha='center', fontweight='bold')
ax.set_title('Shipment Mode Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Shipment Mode')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('outputs/chart1_shipment_mode.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Air shipments dominate at 59% - significant opportunity for cost optimization via modal shift')
print('Business Impact: Positive - identifies over-reliance on expensive air freight')

In [ ]:
# Chart 2: Product Group Distribution
# Why: Identify the most critical product categories
fig, ax = plt.subplots(figsize=(8, 5))
pg = df['Product Group'].value_counts()
ax.pie(pg.values, labels=pg.index, autopct='%1.1f%%', startangle=90)
ax.set_title('Product Group Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/chart2_product_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: ARV products = 82.8% of shipments - requires dedicated pipeline optimization')

In [ ]:
# Chart 3: Freight Cost Distribution
# Why: Understand cost spread and outliers
fig, ax = plt.subplots(figsize=(10, 5))
freight = df['Freight Cost (USD)'].dropna()
ax.hist(freight[freight < freight.quantile(0.95)], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_title('Freight Cost Distribution (USD) - 95th Percentile', fontsize=14, fontweight='bold')
ax.set_xlabel('Freight Cost (USD)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('outputs/chart3_freight_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Right-skewed distribution with high-cost outliers that require investigation')

In [ ]:
# Chart 4: Delivery Delay Distribution
# Why: Measure on-time delivery performance
fig, ax = plt.subplots(figsize=(10, 5))
delay = df['Delivery Delay (Days)'].dropna()
ax.hist(delay[(delay>-100)&(delay<200)], bins=60, color='coral', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='On-time')
ax.set_title('Delivery Delay Distribution (Days)', fontsize=14, fontweight='bold')
ax.set_xlabel('Delay (Days) - Negative=Early, Positive=Late')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/chart4_delay_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Insight: Avg delay={delay.mean():.1f} days | Median={delay.median():.1f} days')

### Bivariate Analysis

In [ ]:
# Chart 5: Shipment Mode vs Avg Freight Cost
# Why: Compare cost-effectiveness across transport modes
fig, ax = plt.subplots(figsize=(9, 5))
avg_cost = df.groupby('Shipment Mode')['Freight Cost (USD)'].mean().sort_values(ascending=False)
bars = ax.bar(avg_cost.index, avg_cost.values, color=sns.color_palette('Set2', len(avg_cost)))
for bar, val in zip(bars, avg_cost.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100, f'${val:,.0f}', ha='center', fontweight='bold')
ax.set_title('Average Freight Cost by Shipment Mode', fontsize=14, fontweight='bold')
ax.set_xlabel('Shipment Mode')
ax.set_ylabel('Avg Freight Cost (USD)')
plt.tight_layout()
plt.savefig('outputs/chart5_cost_by_mode.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Air Charter highest avg cost - minimize where possible')

In [ ]:
# Chart 6: Shipment Mode vs Avg Delivery Delay
# Why: On-time performance comparison
fig, ax = plt.subplots(figsize=(9, 5))
avg_delay = df.groupby('Shipment Mode')['Delivery Delay (Days)'].mean().sort_values()
colors = ['green' if v<=0 else 'red' for v in avg_delay.values]
bars = ax.bar(avg_delay.index, avg_delay.values, color=colors)
ax.axhline(0, color='black', linestyle='--')
for bar, val in zip(bars, avg_delay.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.1f}d', ha='center', fontweight='bold')
ax.set_title('Average Delivery Delay by Shipment Mode', fontsize=14, fontweight='bold')
ax.set_xlabel('Shipment Mode')
ax.set_ylabel('Avg Delay (Days)')
plt.tight_layout()
plt.savefig('outputs/chart6_delay_by_mode.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 7: Top 10 Countries by Shipment Count
# Why: Identify key geographic markets
fig, ax = plt.subplots(figsize=(10, 6))
top_countries = df['Country'].value_counts().head(10)
bars = ax.barh(top_countries.index[::-1], top_countries.values[::-1], color=sns.color_palette('Blues_r', 10))
for bar, val in zip(bars, top_countries.values[::-1]):
    ax.text(bar.get_width()+10, bar.get_y()+bar.get_height()/2, f'{val:,}', va='center')
ax.set_title('Top 10 Countries by Shipment Count', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Shipments')
plt.tight_layout()
plt.savefig('outputs/chart7_top_countries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 8: Line Item Value vs Freight Cost (Scatter)
# Why: Detect correlation between shipment value and cost
fig, ax = plt.subplots(figsize=(10, 6))
sc = df[['Line Item Value','Freight Cost (USD)','Shipment Mode']].dropna()
sc = sc[sc['Line Item Value']<sc['Line Item Value'].quantile(0.95)]
sc = sc[sc['Freight Cost (USD)']<sc['Freight Cost (USD)'].quantile(0.95)]
for mode, grp in sc.groupby('Shipment Mode'):
    ax.scatter(grp['Line Item Value'], grp['Freight Cost (USD)'], label=mode, alpha=0.4, s=10)
ax.set_title('Line Item Value vs Freight Cost', fontsize=14, fontweight='bold')
ax.set_xlabel('Line Item Value (USD)')
ax.set_ylabel('Freight Cost (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/chart8_value_vs_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Weak value-cost correlation suggests pricing is not purely value-based')

In [ ]:
# Chart 9: Vendor INCO Term vs Avg Freight Cost
# Why: Evaluate cost efficiency of different vendor agreements
fig, ax = plt.subplots(figsize=(10, 5))
inco = df.groupby('Vendor INCO Term')['Freight Cost (USD)'].mean().sort_values(ascending=False)
bars = ax.bar(inco.index, inco.values, color=sns.color_palette('Set3', len(inco)))
ax.set_title('Average Freight Cost by Vendor INCO Term', fontsize=14, fontweight='bold')
ax.set_xlabel('Vendor INCO Term')
ax.set_ylabel('Avg Freight Cost (USD)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outputs/chart9_inco_cost.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 10: Product Group vs Line Item Value (Boxplot)
# Why: Compare value distribution across product categories
fig, ax = plt.subplots(figsize=(10, 6))
bdf = df[['Product Group','Line Item Value']].dropna()
bdf = bdf[bdf['Line Item Value']<bdf['Line Item Value'].quantile(0.95)]
sns.boxplot(data=bdf, x='Product Group', y='Line Item Value', ax=ax, palette='Set2')
ax.set_title('Line Item Value by Product Group', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/chart10_value_by_product.png', dpi=150, bbox_inches='tight')
plt.show()

### Multivariate Analysis

In [ ]:
# Chart 11: Correlation Heatmap
# Why: Understand relationships between all numeric variables
fig, ax = plt.subplots(figsize=(10, 7))
num_cols = ['Line Item Quantity','Line Item Value','Pack Price','Unit Price',
            'Weight (Kilograms)','Freight Cost (USD)','Line Item Insurance (USD)',
            'Delivery Delay (Days)','Cost per Unit']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Heatmap - Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/chart11_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Pack Price and Unit Price are highly correlated as expected')

In [ ]:
# Chart 12: Top 10 Countries by Average Delivery Delay
# Why: Pinpoint geographic delivery bottlenecks
fig, ax = plt.subplots(figsize=(10, 6))
top_delay = df.groupby('Country')['Delivery Delay (Days)'].mean().sort_values(ascending=False).head(10)
colors = ['red' if v>30 else 'orange' if v>10 else 'green' for v in top_delay.values]
bars = ax.barh(top_delay.index[::-1], top_delay.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linestyle='--')
ax.set_title('Top 10 Countries by Avg Delivery Delay', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Delay (Days)')
plt.tight_layout()
plt.savefig('outputs/chart12_delay_by_country.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 13: Product Group x Shipment Mode Heatmap
# Why: Multivariate view of product-mode combinations
fig, ax = plt.subplots(figsize=(10, 5))
pivot = df.groupby(['Product Group','Shipment Mode']).size().unstack(fill_value=0)
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Shipment Count: Product Group x Shipment Mode', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/chart13_product_mode_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: ARV via Air dominates - evaluate Ocean alternatives for cost savings')

In [ ]:
# Chart 14: First Line Designation vs Freight Cost (Violin)
# Why: Compare cost distribution for first-line vs non-first-line
fig, ax = plt.subplots(figsize=(9, 5))
vdf = df[['First Line Designation','Freight Cost (USD)']].dropna()
vdf = vdf[vdf['Freight Cost (USD)']<vdf['Freight Cost (USD)'].quantile(0.95)]
sns.violinplot(data=vdf, x='First Line Designation', y='Freight Cost (USD)', palette='Set2', ax=ax)
ax.set_title('Freight Cost: First Line vs Non-First Line', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/chart14_firstline_cost.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 15: Top 10 Manufacturing Sites by Avg Freight Cost
# Why: Identify costly manufacturing locations
fig, ax = plt.subplots(figsize=(12, 6))
top_sites = df.groupby('Manufacturing Site')['Freight Cost (USD)'].mean().sort_values(ascending=False).head(10)
bars = ax.bar(range(len(top_sites)), top_sites.values, color=sns.color_palette('Reds_r', 10))
ax.set_xticks(range(len(top_sites)))
ax.set_xticklabels(top_sites.index, rotation=45, ha='right')
ax.set_title('Top 10 Manufacturing Sites by Avg Freight Cost', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Freight Cost (USD)')
plt.tight_layout()
plt.savefig('outputs/chart15_site_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Certain sites consistently expensive - target for vendor renegotiation')

## 5. Conclusion & Recommendations

### Key Insights Summary

| # | Insight | Business Impact |
|---|---------|----------------|
| 1 | Air shipments = 59% of total | High cost exposure |
| 2 | ARV = 82.8% of shipments | Critical category needing dedicated pipeline |
| 3 | Specific countries show high delays | Route optimization needed |
| 4 | Vendor INCO Terms affect cost significantly | Renegotiation opportunity |
| 5 | Some manufacturing sites are expensive | Vendor consolidation potential |

### Recommendations
1. **Modal Shift** — Move 20% of Air shipments to Ocean/Truck to cut freight costs by 15-25%
2. **ARV Pipeline** — Create dedicated supply chain for ARV (82.8% of shipments)
3. **High-Delay Countries** — Reroute or change shipment mode to meet delivery SLAs
4. **Vendor Renegotiation** — Target expensive INCO Terms and manufacturing sites
5. **Predictive Monitoring** — Build delay prediction model using Delivery Delay feature